# 🛒 Superstore — Pipeline Data Engineering
**Stack :** Python · pandas · SQLAlchemy · PostgreSQL

> Ce notebook lit le fichier CSV, prépare les données, les découpe en tables normalisées, puis les insère dans PostgreSQL.

---
## Cellule 1 — Importation des bibliothèques

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

---
## Cellule 2 — Chargement du dataset

In [ ]:
df = pd.read_csv("a.csv")

df.head()

---
## Cellule 3 — Nettoyage de base

In [ ]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"]  = pd.to_datetime(df["ship_date"])

print("Nettoyage terminé")

---
## Cellule 4 — Table `regions`

In [ ]:
regions = df[["postal_code", "city", "state", "region", "country"]].copy()

regions = regions.drop_duplicates(subset=["postal_code"])
regions = regions.reset_index(drop=True)

print("Nombre de régions :", len(regions))
regions.head()

---
## Cellule 5 — Table `customers`

In [ ]:
customers = df[["customer_id", "customer_name", "segment", "postal_code"]].copy()

customers = customers.drop_duplicates(subset=["customer_id"])
customers = customers.reset_index(drop=True)

print("Nombre de clients :", len(customers))
customers.head()

---
## Cellule 6 — Table `categories`

In [ ]:
categories = df[["category"]].drop_duplicates().copy()
categories = categories.reset_index(drop=True)

categories.index = categories.index + 1
categories = categories.reset_index()
categories.columns = ["category_id", "category_name"]

print("Catégories :")
categories

---
## Cellule 7 — Table `products`

In [ ]:
products = df[["product_id", "product_name", "sub_category", "category"]].copy()
products = products.drop_duplicates(subset=["product_id"])

products = products.merge(
    categories.rename(columns={"category_name": "category"}),
    on="category",
    how="left"
)

products = products.drop(columns=["category"])
products = products.reset_index(drop=True)

print("Nombre de produits :", len(products))
products.head()

---
## Cellule 8 — Table `orders`

In [ ]:
orders = df[["order_id", "order_date", "ship_date", "ship_mode", "customer_id"]].copy()

orders = orders.drop_duplicates(subset=["order_id"])
orders = orders.reset_index(drop=True)

print("Nombre de commandes :", len(orders))
orders.head()

---
## Cellule 9 — Table `order_details`

In [ ]:
order_details = df[["order_id", "product_id", "sales", "cost", "profit"]].copy()

order_details = order_details.drop_duplicates(subset=["order_id", "product_id"])
order_details = order_details.reset_index(drop=True)

print("Nombre de lignes de détail :", len(order_details))
order_details.head()

---
## Cellule 10 — Connexion à PostgreSQL

In [ ]:
password = "1234"
host     = "localhost"
port     = "5432"
database = "superstore_db"

engine = create_engine(
    f"postgresql+psycopg2://postgres:{password}@{host}:{port}/{database}",
    connect_args={"client_encoding": "utf8"}
)

print(" Connexion à PostgreSQL réussie !")

---
## Cellule 11 — Insertion des données dans PostgreSQL

In [ ]:
with engine.begin() as conn:

    regions.to_sql("regions", conn, if_exists="append", index=False)
    print(" regions       —", len(regions), "lignes insérées")

    customers.to_sql("customers", conn, if_exists="append", index=False)
    print(" customers     —", len(customers), "lignes insérées")

    categories.to_sql("categories", conn, if_exists="append", index=False)
    print(" categories    —", len(categories), "lignes insérées")

    products.to_sql("products", conn, if_exists="append", index=False)
    print(" products      —", len(products), "lignes insérées")

    orders.to_sql("orders", conn, if_exists="append", index=False)
    print(" orders        —", len(orders), "lignes insérées")

    order_details.to_sql("order_details", conn, if_exists="append", index=False)
    print(" order_details —", len(order_details), "lignes insérées")
